# Environment Setup

In [6]:
!pip install wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 196.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.2/208.2 kB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 138.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 225.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.7/356.7 kB 134.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 32.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [7]:
%%capture
%pip install -U bitsandbytes
%pip install -U peft
%pip install -U accelerate
%pip install -U trl
%pip install -U datasets

In [8]:
%pip install datasets==2.16.0

  Using cached datasets-2.16.0-py3-none-any.whl.metadata (20 kB)
Using cached datasets-2.16.0-py3-none-any.whl (507 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.19.1 requires datasets>=3.0.0, but you have datasets 2.16.0 which is incompatible.

[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Imports

In [9]:
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from random import randrange
from peft import LoraConfig, get_peft_model, AutoPeftModelForCausalLM
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments,BitsAndBytesConfig
from trl import SFTTrainer

## Wandb

In [10]:
#wandb_key="7377fc8b6b7b55dd9f6dc8b9aaf9e98cf18b64cf"

In [ ]:
import wandb

# Log in to your W&B account
wandb.login()

# Initialize a new W&B run
wandb.init(
    project="sft-llama3.1-8b-instruct-device-zero-with-ocr-qa"
)

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

  ········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abrahamkaikobadtushar (axiler) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Model & Tokenizer Loading

In [12]:
from huggingface_hub import login
import os

token = "HF_TOKEN_REDACTED"
os.environ['HF_TOKEN'] = token

login(token=token)
print("Logged in to Hugging Face!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face!


In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

# Dataset load and preprocess

In [28]:
import pandas as pd
device_train=pd.read_csv("device_train.csv")

In [29]:
device_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2942 entries, 0 to 2941
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Unnamed: 0       2942 non-null   int64 
 1   title            2942 non-null   object
 2   body             2942 non-null   object
 3   tags             2942 non-null   object
 4   link             2942 non-null   object
 5   score            2942 non-null   int64 
 6   creation_date    2942 non-null   object
 7   answer_count     2942 non-null   int64 
 8   comments         2942 non-null   object
 9   answers          2942 non-null   object
 10  selected_answer  2942 non-null   object
 11  images           2942 non-null   object
dtypes: int64(3), object(9)
memory usage: 275.9+ KB


## ocr load

In [ ]:
import json

with open("device-train-img-ocr-response-filtered.jsonl", "r", encoding="utf-8") as file:
    device_train_ocr = [json.loads(line) for line in file]

In [24]:
from typing import List, Dict, Any

def ocr_extract(data) -> List[str]:
    results: List[str] = []

    for entry in data:
        lines=[]

        for img_i, img in enumerate(entry.get("images", [])):
            lines.append(f"img{img_i}:")

            for resp in img.get("response", []):
                if isinstance(resp, dict):
                    # --- non-table text ---
                    ntt = resp.get("non_table_text")
                    if ntt:
                        lines.append("Non-table text:")
                        lines.extend(ntt if isinstance(ntt, list) else [str(ntt)])

                    # --- tables ---
                    tables = resp.get("table")
                    if tables:
                        lines.append("Tables:")
                        for t_i, tbl in enumerate(tables if isinstance(tables, list) else [tables]):
                            lines.append(f"Table {t_i}:")
                            lines.append(tbl.get("content", "") if isinstance(tbl, dict) else str(tbl))
                    continue

                # Unknown or malformed type
                lines.append(f"[WARN] Unhandled response type: {type(resp).__name__}")
                lines.append(str(resp))

        formatted = "\n".join(lines).strip()
        if formatted:
            results.append(formatted)

    return results

In [25]:
# format ocr response for prompt
device_train_ocr_response_format = ocr_extract(device_train_ocr)   

In [30]:
len(device_train_ocr_response_format),device_train.shape

(2942, (2942, 12))

In [40]:
device_train['images'][1793]

"['https://i.sstatic.net/dtg8O.png']"

In [41]:
print(device_train_ocr_response_format[1793])

img0:
Non-table text:
                        Logging call in user
code
      Logger flow
                                                       Handler flow
                                                                        LogRecord passed to
handler
                        e.g.
logger.info(...)
                  Logger enabled for
level of call?
                                No
                                                                                 No
                                                                  Handler enabled for
level of LogRecord?
                                        Stop
                                                                                        Stop
                       Yes
                                                                        Yes
                     Create
                    LogRecord
                                                                  Does a filter attached
to handler reject the
record?
 

## processed

In [42]:
device_train=device_train[['title','body','selected_answer','images']]

In [43]:
device_train['context'] = "title: " + device_train['title'].astype(str) + "\nbody: " + device_train['body'].astype(str).str.strip()

In [ ]:
device_train['selected_answer']=device_train['selected_answer'].str.strip()

In [44]:
device_train_processed=pd.DataFrame(
    {
        'question':device_train['context'],
        'answer':device_train['selected_answer']
    }
)

In [47]:
device_train_processed['ocr']=device_train_ocr_response_format

In [48]:
device_train_processed.head(2)

,question,answer,ocr
0,title: WireShark Remote Capture failed:NFLOG l...,The error message is explaining what is the ca...,img0:\nNon-table text:\n En...
1,title: Are there events which show application...,\r\nIn windows you can set the policy of secur...,img0:\nTables:\nTable 0:\n+-------------------...


In [17]:
"""cloud_train_processed=cloud_train_processed[:500]
cloud_val_processed=cloud_val_processed[:100]"""

'cloud_train_processed=cloud_train_processed[:500]\ncloud_val_processed=cloud_val_processed[:100]'

## prompt

In [49]:
zero_shot = '''
You are an expert device assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
{}

Question:
{}

Answer:
{}
'''

def formatting_func(row):
    return zero_shot.format(row['ocr'],row["question"],row["answer"])

device_train_processed["text"] = device_train_processed.apply(formatting_func, axis=1)

In [50]:
print(device_train_processed["text"][3])


You are an expert device assistant. Provide a concise answer to the following question based on the ocr-extracted text from the given image.
If you lack sufficient information to answer, respond with "No answer."

OCR Text:
img0:
Tables:
Table 0:
+-----------------------+-----------------+
| Volume]               | 237.46 GB       |
| Size:                 |                 |
+-----------------------+-----------------+
| BitLocker Version:    | 2.0             |
+-----------------------+-----------------+
| Conversion Status:    | Fully Encrypted |
+-----------------------+-----------------+
| Percentage Encrypted: | 100.0%          |
+-----------------------+-----------------+
| Encryption Method:    | XTS-AES 128     |
+-----------------------+-----------------+
| Protection Status:    | Protection Off  |
+-----------------------+-----------------+
| Lock Status:          | Unlocked        |
+-----------------------+-----------------+
| Identification Field: | Unknown         |
+---

In [51]:
# Split 20% for validation, 80% for training
val_df = device_train_processed[["text"]].sample(frac=0.2, random_state=42)
train_df = device_train_processed[["text"]].drop(val_df.index)

In [52]:
from datasets import Dataset, load_dataset

# Convert DataFrame to HF Dataset with just the 'text' column
trainset = Dataset.from_pandas(train_df.reset_index(drop=True))
valset = Dataset.from_pandas(val_df.reset_index(drop=True))

print(f"Training set size: {len(trainset)}")
print(f"Validation set size: {len(valset)}")

Training set size: 2354
Validation set size: 588


In [53]:
trainset,valset


(Dataset({
     features: ['text'],
     num_rows: 2354
 }),
 Dataset({
     features: ['text'],
     num_rows: 588
 }))

## Testing

In [ ]:
# Test
sample_input = device_train_processed["text"].iloc[1]

In [ ]:
inputs = tokenizer(
    sample_input,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(model.device)

model.eval()
with torch.no_grad():
    output_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
        eos_token_id=tokenizer.eos_token_id
    )

output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(output_text)

# Training Arguments

In [ ]:
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
# training args
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="sft-llama3.1-8b-instruct-device-zero-with-ocr-qa",
    num_train_epochs=5,
    per_device_train_batch_size=3,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=50,
    gradient_checkpointing=True,
    seed=42,
    push_to_hub=True,
    load_best_model_at_end=True,
    hub_model_id="Bakugo123/sft-llama3.1-8b-instruct-device-zero-with-ocr-qa",
    hub_strategy="end",
)

# Trainer

In [56]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

In [57]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=trainset,
    eval_dataset=valset,
    peft_config=peft_config,
    #tokenizer=tokenizer,
    args=args,
    #max_seq_length=1024,
    #dataset_text_field="text"
)

Adding EOS to train dataset:   0%|          | 0/2354 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2354 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2354 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/588 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/588 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/588 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
# Train
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
15,No log,2.233028


In [ ]:
# Evaluate the model
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

In [ ]:
# Save the model and tokenizer
#trainer.save_model("sft-llama3-8b-instruct-cloud-zero-with-ocr-qa")
#tokenizer.save_pretrained("sft-llama3-8b-instruct-cloud-zero-with-ocr-qa")

# Save Fine Tune Model

In [ ]:
#model.push_to_hub("Bakugo123/sft-llama3-8b-instruct-cloud-zero-with-ocr-qa")
#tokenizer.push_to_hub("Bakugo123/sft-llama3-8b-instruct-cloud-zero-with-ocr-qa")

In [ ]:
trainer.push_to_hub()

In [ ]:
torch.cuda.empty_cache()

## Load fine tune model

In [ ]:
#model_id="Bakugo123/model_name" #fine tune model id

In [ ]:
"""from transformers import pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto",
)"""

In [ ]:
"""question = ""
messages = [{"role": "user", "content": question}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=120, do_sample=True, repetition_penalty=1.2)
print(outputs[0]["generated_text"])
"""